In [ ]:
from pathlib import Path
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import os
import re
from collections import defaultdict
from skimage.measure import regionprops, regionprops_table
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
import sys
import io
from skimage.io import imread
import seaborn as sns
import matplotlib.ticker as ticker
%matplotlib inline

# makes figures look better in Jupyter
sns.set_context('talk')
sns.set_style("ticks")
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

In [ ]:
df_long = pd.read_csv('/Volumes/salmonella/users/madison/2025_final_inf_analysis/Filtered_Final_Infection_Data.csv')

In [ ]:
#Validating that the segmentation never picked up a "bacteria" in my infection negative controls
empty_wells = df_long[df_long['SL_strain'].isnull()]
bacteria_detected = empty_wells[empty_wells['bact_area'].notnull()]
bacteria_detected

In [ ]:
#Plotting distribution of bacterial label median intensities
times = [2.5, 3.5, 4.5, 5.5, 6.5]
strains = ['pLL565'] #Indicates reporter strain, removes GFP(-) control (pLL889)
dose = [0]
plot_df = df_long[df_long['SL_strain'].isin(strains) & df_long['hpi'].isin(times)& df_long['condition'].isin(dose)]
sns.kdeplot(data=plot_df, x='bact_GFP_median_intensity', hue='hpi', palette = 'viridis', log_scale=True)
sns.despine()
plt.axvline(634.325, color='gray', linestyle='--', linewidth=1, label=f'Threshold = 634.325') #set based on GFP(-) control, used to calculate # of bact label pixels per label that fall above threshold during extraction. 
#plt.savefig(output_dir +'/kde_untreated.pdf')

In [ ]:
#Plotting distribution of bacterial label median intensities
times = [2.5, 3.5, 4.5, 5.5, 6.5]
strains = ['pLL889'] #Plotting GFP(-) SL control to show this intensity stays in the 'off' peak
dose = [0]
plot_df = df_long[df_long['SL_strain'].isin(strains) & df_long['hpi'].isin(times)& df_long['condition'].isin(dose)]
sns.kdeplot(data=plot_df, x='bact_GFP_median_intensity', hue='hpi', palette = 'viridis', log_scale=True)
sns.despine()
plt.axvline(634.325, color='gray', linestyle='--', linewidth=1, label=f'Threshold = 634.325')
#plt.savefig(output_dir +'/kde_untreated.pdf')

In [ ]:
#Plot same thing for the reporter as above but by dose of BAF-A1
times = [2.5, 3.5, 4.5, 5.5, 6.5]
strains = ['pLL565']
plot_df = df_long[df_long['SL_strain'].isin(strains) & df_long['hpi'].isin(times)]
doses = [0, 5, 10, 20, 100]

fig, axes = plt.subplots(5, 1, figsize=(8, 3.75 * len(doses)), sharey=False, sharex=True)
axes = axes.flatten()

for ax, dose in zip(axes, doses):
    data = plot_df[plot_df['condition']==dose]  # shape: (cell,)
    sns.kdeplot(data=data, x='bact_GFP_median_intensity', hue='hpi', palette = 'viridis', log_scale=True, legend = False, ax=ax)
    ax.axvline(634.325, color='gray', linestyle='--', linewidth=1)
    
plt.tight_layout()
sns.despine()
#plt.savefig(output_dir +'/all_doses_kde_full.pdf')

In [ ]:
#Showing mRuby intensity from the reporter remains unimodal
times = [2.5, 3.5, 4.5, 5.5, 6.5]
strains = ['pLL565']
dose = [0, 5, 10, 20, 100]
plot_df = df_long[df_long['SL_strain'].isin(strains) & df_long['hpi'].isin(times)& df_long['condition'].isin(dose)]
sns.kdeplot(data=plot_df, x='bact_TRITC_median_intensity', hue='hpi', palette = 'viridis', log_scale=True)
sns.despine()
#plt.savefig(output_dir +'/kde_allconditions_TRITC.pdf')

In [ ]:
#Grouping labels by condition, experiment, hours, and SL_strain to get summary metrics
grouped_df =  df_long.groupby(['condition', 'experiment', 'hpi', 'SL_strain'], as_index=False).agg({
        'bact_GFP_count_above_threshold': 'sum', #sum of all bacterial pixels above GFP threshold
        'bact_area': 'sum', #sum of bacterial label area
        'bact_GFP_median_intensity':'mean', #Mean of the median label reporter intensities
        'bact_TRITC_median_intensity':'mean', #Mean of the median label 
        'bact_Cy5_median_intensity': 'count'
        
    })
grouped_df['pos/area']= grouped_df['bact_GFP_count_above_threshold']/grouped_df['bact_area'] #Getting the SPI-2 fraction on metric
grouped_df.rename(columns = {"bact_Cy5_median_intensity":"label_count"}, inplace=True)
grouped_df


In [ ]:
#Plot SPI-2 positive pixels per dose across experiments
palette = {0:'lightgrey', 5:'lightsteelblue', 10:'steelblue', 20:'mediumblue', 100:'navy'}
df_plot = grouped_df[grouped_df['SL_strain']=='pLL565']
ax = sns.lineplot(data = df_plot, x='hpi', y='pos/area', hue='condition', palette = palette, linewidth = 2.5, errorbar = 'sd',  legend = True)
sns.despine()
ax.set_xticks([1, 2, 3, 4, 5, 6, 7])
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.) 
#plt.savefig(output_dir +'/full_lineplot.pdf')

In [ ]:
#Plot intensity of mRuby grouped the same way
palette = {0:'lightgrey', 5:'lightsteelblue', 10:'steelblue', 20:'mediumblue', 100:'navy'}
df_plot = grouped_df[grouped_df['SL_strain']=='pLL565']
ax = sns.lineplot(data = df_plot, x='hpi', y='bact_TRITC_median_intensity', hue='condition', palette = palette, linewidth = 2.5, errorbar = 'sd',  legend = True)
sns.despine()
ax.set_xticks([1, 2, 3, 4, 5, 6, 7])
plt.yscale('log')
plt.ylim(1000, 10000)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.) 
#plt.savefig(output_dir +'/full_TRITC_meanint_lineplot.pdf')

In [ ]:
#Plot bact label area
palette = {0:'lightgrey', 5:'lightsteelblue', 10:'steelblue', 20:'mediumblue', 100:'navy'}
df_plot = grouped_df[grouped_df['SL_strain']=='pLL565']
ax = sns.lineplot(data = df_plot, x='hpi', y='bact_area', hue='condition', palette = palette, linewidth = 2.5, errorbar = 'sd',  legend = True)
sns.despine()
ax.set_xticks([1, 2, 3, 4, 5, 6, 7])

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.) 
#plt.savefig(output_dir +'/full_TRITC_total_area_lineplot.pdf')

In [ ]:
#Plot SPI-2 pos pixels by replicate
palette = {0:'lightgrey', 5:'lightsteelblue', 10:'steelblue', 20:'mediumblue', 100:'navy'}
df_plot = grouped_df[grouped_df['SL_strain']=='pLL565']
experiments = [20251024, 20251027, 20251030, 20251105]
nrows = 2
ncols = 2

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows), sharey=True, sharex=True)
axes = axes.flatten()

for ax, e in zip(axes, experiments):
    data = df_plot[df_plot['experiment']==e]  # shape: (cell,)
    sns.lineplot(data = data, x='hpi', y='pos/area', hue='condition', palette = palette, linewidth = 2.5, errorbar='sd', legend = False, ax=ax)
    ax.set_xticks([1, 2, 3, 4, 5, 6, 7])
    ax.set_yticks([0, 0.4, 0.8])
    

plt.tight_layout()
sns.despine()
#plt.savefig(output_dir +'/rep_sep_lineplot.pdf')


In [ ]:
#Label area by replicate
palette = {0:'lightgrey', 5:'lightsteelblue', 10:'steelblue', 20:'mediumblue', 100:'navy'}
df_plot = grouped_df[grouped_df['SL_strain']=='pLL565']
experiments = [20251024, 20251027, 20251030, 20251105]
nrows = 2
ncols = 2

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows), sharey=True, sharex=True)
axes = axes.flatten()

for ax, e in zip(axes, experiments):
    data = df_plot[df_plot['experiment']==e]  # shape: (cell,)
    sns.lineplot(data = data, x='hpi', y='bact_area', hue='condition', palette = palette, linewidth = 2.5, errorbar='sd', legend = False, ax=ax)
    ax.set_xticks([1, 2, 3, 4, 5, 6, 7])
    

plt.tight_layout()
sns.despine()
#plt.savefig(output_dir +'/rep_sep_area_lineplot.pdf')

In [ ]:
#Label count means
palette = {0:'lightgrey', 5:'lightsteelblue', 10:'steelblue', 20:'mediumblue', 100:'navy'}
df_plot = grouped_df[grouped_df['SL_strain']=='pLL565']
ax = sns.lineplot(data = df_plot, x='hpi', y='label_count', hue='condition', palette = palette, linewidth = 2.5, errorbar = 'sd',  legend = True)
sns.despine()
ax.set_xticks([1, 2, 3, 4, 5, 6, 7])

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.) 
#plt.savefig(output_dir +'/full_count_lineplot.pdf')

In [ ]:
#Label count by replicate
palette = {0:'lightgrey', 5:'lightsteelblue', 10:'steelblue', 20:'mediumblue', 100:'navy'}
df_plot = grouped_df[grouped_df['SL_strain']=='pLL565']
experiments = [20251024, 20251027, 20251030, 20251105]
nrows = 2
ncols = 2

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows), sharey=True, sharex=True)
axes = axes.flatten()

for ax, e in zip(axes, experiments):
    data = df_plot[df_plot['experiment']==e]  # shape: (cell,)
    sns.lineplot(data = data, x='hpi', y='label_count', hue='condition', palette = palette, linewidth = 2.5, errorbar='sd', legend = False, ax=ax)
    ax.set_xticks([1, 2, 3, 4, 5, 6, 7])
    ax.set_yticks([0, 50, 100, 150, 200, 250, 300])


plt.tight_layout()
sns.despine()
#plt.savefig(output_dir +'/rep_sep_label_count_lineplot.pdf')

In [ ]:
#Function to pulling out time of an experiment group to cross the threshold of a given fraction on 
def threshold_crossing(df, threshold):
    mask = df['pos/area'] > threshold

    idx = (
        mask.groupby([df['condition'], df['experiment'], df['SL_strain']]).apply(lambda x: x.idxmax() if x.any() else np.nan)
        .dropna())

    return (df.loc[idx.astype(int), ['condition', 'experiment', 'SL_strain', 'hpi']].assign(threshold=threshold).reset_index(drop=True))


In [ ]:
#Applying that thresholding function to get a summary metric for time to a fraction of 0.15 'ON'
thresholds = [0.15]

th_cross = pd.concat(
    [threshold_crossing(grouped_df, t) for t in thresholds],
    ignore_index=True)

In [ ]:
#Plotting time to cross that threshold for each replicate
plot_data = th_cross[th_cross['SL_strain']=='pLL565']
plt.figure(figsize=(3.75, 3))
sns.barplot(data=plot_data, x='condition', y='hpi', hue='condition', errorbar = 'sd', palette = palette)
sns.swarmplot(data=plot_data, x='condition', y='hpi', hue='condition', palette = palette, dodge=False, legend = False, edgecolor='white', linewidth = 1)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.) 
sns.despine()
#plt.savefig(output_dir +'0.15_time_bar.pdf')

In [ ]:
#performing a ttest on this
from scipy.stats import ttest_ind

plot_data['condition'] = plot_data['condition'].astype(str)
groups = plot_data.groupby(['condition'])['hpi'].apply(list).to_dict()

results = []

# Store the results
results = []

# Get the keys (group identifiers) and convert to a list for easy pairing
group_keys = list(groups.keys())

# Perform t-tests between each pair of groups within the same Timepoint
for i in range(len(group_keys)):
    for j in range(i + 1, len(group_keys)):
        group1, group2 = group_keys[i], group_keys[j]
        data1, data2 = groups[group1], groups[group2]
        if len(data1) > 1 and len(data2) > 1:
            t_stat, p_val = ttest_ind(data1, data2)
            results.append({
                    'Group 1': group1,
                    'Group 2': group2,
                    't-statistic': t_stat,
                    'p-value': p_val
                })

# Convert results to DataFrame for better visualization
results_df = pd.DataFrame(results)
#results_df.to_csv(output_dir + '/threshold_pvalues.csv')

In [ ]:
#Creating a table of slopes across time
df_slope = grouped_df[grouped_df['SL_strain'] == 'pLL565']

df_slope = (df_slope.sort_values(by=['condition', 'experiment', 'hpi']).reset_index(drop=True))

df_slope['rolling_ratio'] = (df_slope.groupby(['condition', 'experiment'])['pos/area'].transform(lambda x: x.rolling(window=12, min_periods=1).mean()))

df_slope['1-RR'] = 1 - df_slope['rolling_ratio']

In [ ]:
#Identifying max slope
df_slope['slope'] = (df_slope.sort_values(['condition', 'experiment', 'hpi']).groupby(['condition', 'experiment']).apply(lambda g: g['rolling_ratio'].diff() / g['hpi'].diff()).reset_index(level=[0,1], drop=True))

idx = df_slope.groupby(['condition', 'experiment'])['slope'].idxmax()
max_slopes = df_slope.loc[idx].reset_index(drop=True)
max_slopes

In [ ]:
#plot max slopes
plt.figure(figsize=(3.75, 3))
sns.barplot(data=max_slopes, x='condition', y='slope', hue='condition', errorbar = 'sd', palette = palette)
sns.swarmplot(data=max_slopes, x='condition', y='slope', hue='condition', palette = palette, dodge=False, legend = False, edgecolor='white', linewidth = 1)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.) 
sns.despine()
#plt.savefig(output_dir +'/max_slopes_bar_2.pdf')

In [ ]:
#Run ttest on slopes
from scipy.stats import ttest_ind
max_slopes['condition'] = max_slopes['condition'].astype(str)
columns_to_keep = ['condition', 'slope']
max_slopes_ttest = max_slopes[columns_to_keep]

groups = max_slopes_ttest.groupby(['condition'])['slope'].apply(list).to_dict()

# Store the results
results = []

# Get the keys (group identifiers) and convert to a list for easy pairing
group_keys = list(groups.keys())

# Perform t-tests between each pair of groups within the same Timepoint
for i in range(len(group_keys)):
    for j in range(i + 1, len(group_keys)):
        group1, group2 = group_keys[i], group_keys[j]
        data1, data2 = groups[group1], groups[group2]
        if len(data1) > 1 and len(data2) > 1:
            t_stat, p_val = ttest_ind(data1, data2)
            results.append({
                    'Group 1': group1,
                    'Group 2': group2,
                    't-statistic': t_stat,
                    'p-value': p_val
                })

# Convert results to DataFrame for better visualization
results_df = pd.DataFrame(results)
#results_df.to_csv(output_dir + '/max_slopes_pvalues.csv')
results_df

In [ ]:
#Ttest on times to max slope
max_slopes['condition'] = max_slopes['condition'].astype(str)
columns_to_keep = ['condition', 'hpi']
max_slopes_ttest = max_slopes[columns_to_keep]

groups = max_slopes_ttest.groupby(['condition'])['hpi'].apply(list).to_dict()

# Store the results
results = []

# Get the keys (group identifiers) and convert to a list for easy pairing
group_keys = list(groups.keys())

# Perform t-tests between each pair of groups within the same Timepoint
for i in range(len(group_keys)):
    for j in range(i + 1, len(group_keys)):
        group1, group2 = group_keys[i], group_keys[j]
        data1, data2 = groups[group1], groups[group2]
        if len(data1) > 1 and len(data2) > 1:
            t_stat, p_val = ttest_ind(data1, data2)
            results.append({
                    'Group 1': group1,
                    'Group 2': group2,
                    't-statistic': t_stat,
                    'p-value': p_val
                })

# Convert results to DataFrame for better visualization
results_df = pd.DataFrame(results)
#results_df.to_csv(output_dir + '/max_slopes_times_pvalues.csv')

In [ ]:
idx = df_slope.groupby(['condition', 'experiment'])['rolling_ratio'].idxmax()
max_values = df_slope.loc[idx].reset_index(drop=True)
max_values

In [ ]:
#plot max values
plt.figure(figsize=(3.75, 3))
sns.barplot(data=max_values, x='condition', y='rolling_ratio', hue='condition', errorbar = 'sd', palette = palette)
sns.swarmplot(data=max_values, x='condition', y='rolling_ratio', hue='condition', palette = palette, dodge=False, legend = False, edgecolor='white', linewidth = 1)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.) 
sns.despine()
#plt.savefig(output_dir +'/max_values_bar.pdf')

In [ ]:
#plot time of max values
palette = {0:'lightgrey', 5:'lightsteelblue', 10:'steelblue', 20:'mediumblue', 100:'navy'}

plt.figure(figsize=(3.75, 3))
sns.barplot(data=max_values, x='condition', y='hpi', hue='condition', errorbar = 'sd', palette = palette)
sns.swarmplot(data=max_values, x='condition', y='hpi', hue='condition', palette = palette, dodge=False, legend = False, edgecolor='white', linewidth = 1)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.) 
sns.despine()
#plt.savefig(output_dir +'/max_values_time_bar.pdf')

In [ ]:
#Plot smoothed data
ax = sns.lineplot(data = df_slope, x='hpi', y='rolling_ratio', hue='condition', palette = palette, linewidth = 2.5, errorbar = 'sd',  legend = True)
sns.despine()
ax.set_xticks([1, 2, 3, 4, 5, 6, 7])
ax.set_ylim(0, 0.7)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.) 
#plt.savefig(output_dir+'/smoothed_pooled.pdf')